In [1]:
import pandas as pd 
import numpy as np
from dk_model import DeepKrigingTrainer

In [3]:
deposit_data = pd.read_csv("C:/Users/lin236/Documents/MEC_PC_Data/CSIRO Share/Export 19032024/Output/filtered_deposit_data.csv", low_memory=False)
# Specify the columns to move to the front
columns_to_front = ['X', 'Y', 'Z']

# Reorder the columns
deposit_data = deposit_data[columns_to_front + [col for col in deposit_data.columns if col not in columns_to_front]]
deposit_data

,X,Y,Z,Au_ppm,As_ppm_BESTEL,Hg_ppm_BESTEL,Tl_ppm_BESTEL,Te_ppm_BESTEL,GEOLOGY_Drc,GEOLOGY_DSO,GEOLOGY_SDb,GEOLOGY_Tc
0,0.233019,0.671331,0.270477,0.007147,0.650077,0.197552,0.193853,0.001116,1,0,0,0
1,0.233238,0.671295,0.269044,0.022344,0.650077,0.197552,0.193853,0.001116,1,0,0,0
2,0.233459,0.671257,0.267611,0.024817,0.145289,0.209790,0.055556,0.008990,0,0,1,0
3,0.233680,0.671219,0.266179,0.020370,0.145289,0.209790,0.055556,0.008990,0,0,1,0
4,0.233901,0.671181,0.264747,0.026850,0.145289,0.209790,0.055556,0.008990,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
1214,0.096584,0.891018,0.179664,0.008181,0.105450,0.078671,0.079196,0.000529,0,0,1,0
1215,0.096574,0.890910,0.178218,0.002687,0.105450,0.078671,0.079196,0.000529,0,0,1,0
1216,0.096565,0.890802,0.176772,0.002319,1.000000,0.031469,0.052009,0.000881,0,0,1,0
1217,0.096555,0.890694,0.175326,0.001463,1.000000,0.031469,0.052009,0.000881,0,0,1,0


In [9]:
#reduce df
deposit_data = deposit_data[:2000]

In [4]:
N = len(deposit_data)

lon = deposit_data.values[:, 1]
lat = deposit_data.values[:, 2]
az = deposit_data.values[:, 3]

num_basis_3_lvl = [10**3, 19**3, 37**3]
num_basis_2_lvl = [10**3, 19**3]
num_basis_1_lvl = [10**3]

'''
num_basis_3_lvl = [5**3, 10**3, 18**3]
num_basis_2_lvl = [5**3, 10**3]
num_basis_1_lvl = [5**3]
'''
num_basis_list = [num_basis_3_lvl, num_basis_2_lvl, num_basis_1_lvl]

phi_arrays = []  

# For each grid
for grid in num_basis_list:
    knots_1dx = [np.linspace(0, 1, int(i**(1/3)) + 1) for i in grid]
    knots_1dy = [np.linspace(0, 1, int(i**(1/3)) + 1) for i in grid]
    knots_1dz = [np.linspace(0, 1, int(i**(1/3)) + 1) for i in grid]
    basis_size = 0
    phis = np.zeros((N, sum(grid)))
    
    # For each level of resolution
    for res in range(len(grid)):
        theta = 1 / (grid[res]**(1/3)) * 2.5
        knots_x, knots_y, knots_z = np.meshgrid(knots_1dx[res], knots_1dy[res], knots_1dz[res])
        knots = np.column_stack((knots_x.flatten(), knots_y.flatten(), knots_z.flatten()))
        
        # For each node in the grid
        for i in range(grid[res]):
            d = np.linalg.norm(np.vstack((lon, lat, az)).astype(float).T - knots[i, :], axis=1) / theta
            
            # For each distance of our data to the node i, calculate Wendland kernel
            for j in range(len(d)):
                if 0 <= d[j] <= 1:
                    phis[j, i + basis_size] = (1 - d[j])**6 * (35 * d[j]**2 + 18 * d[j] + 3) / 3
                else:
                    phis[j, i + basis_size] = 0
        
        basis_size += grid[res]
    
    phi_arrays.append(phis)  # Store the phi array for this grid level

# Unpack phi arrays into individual variables
phi_1_lvl, phi_2_lvl, phi_3_lvl = phi_arrays


In [6]:
phis = [phi_1_lvl, phi_2_lvl, phi_3_lvl]
phi_reduces = {}
dfs = []

#phi_columns = deposit_data.columns[10:].tolist()
phi_columns = deposit_data.columns[12:].tolist()

# Display the list of column names
#print(phi_columns[:10])
print(phi_columns[:12])

#total_columns = ['CP_Total', 'PO_Total', 'PY_Total']
#total_columns = ['As_ppm_BESTEL', 'Hg_ppm_BESTEL']

# All covariates
#covariates = total_columns[:3] + ['RQD_Pct', 'Cr_ppm'] 
covariates = ['As_ppm_BESTEL', 'Hg_ppm_BESTEL', 'Tl_ppm_BESTEL', 'Tl_ppm_BESTEL']
# Find column names starting with 'GEOLOGY'
geology_columns = [col for col in deposit_data.columns if col.startswith('GEOLOGY')]

# Print the list of column names
print(geology_columns)

deposit_data = deposit_data.dropna(subset=['Au_ppm'] + covariates + geology_columns + phi_columns)

for idx, phi in enumerate(phis, start=1):
    idx_zero = np.array([], dtype=int)
    for i in range(phi.shape[1]):
        if np.sum(phi[:, i] != 0) == 0:
            idx_zero = np.append(idx_zero, int(i))

    phi_reduce = np.delete(phi, idx_zero, 1)
    phi_reduces[f"phi_{idx}_lvl_reduce"] = phi_reduce
    
    len_phi_regular = phi.shape[1]
    df_phi_regular = pd.DataFrame(phi, columns=[f'phi_{i}' for i in range(len_phi_regular)])
    dfs.append(df_phi_regular)
    
    len_phi_reduce = phi_reduce.shape[1]
    df_phi_reduce = pd.DataFrame(phi_reduce, columns=[f'phi_{i}' for i in range(len_phi_reduce)])
    dfs.append(df_phi_reduce)
    


[]
['GEOLOGY_Drc', 'GEOLOGY_DSO', 'GEOLOGY_SDb', 'GEOLOGY_Tc']


In [7]:
deposit_data_list = []
for df in dfs:
    df_reset = df.reset_index(drop=True)
    deposit_data_reset = deposit_data.reset_index(drop=True)

    # Concatenate along columns
    deposit_data_basis = pd.concat([deposit_data_reset, df], axis=1)
    phi_columns = deposit_data_basis.columns[12:].tolist()
    #total_columns = ['CP_Total','PO_Total', 'PY_Total']
    #total_columns = ['As_ppm_BESTEL', 'Hg_ppm_BESTEL']
    #covariates = total_columns[:3] + ['RQD_Pct', 'Cr_ppm'] 
    covariates = ['As_ppm_BESTEL', 'Hg_ppm_BESTEL']
    #covariates = total_columns[-2:]
    deposit_data_basis = deposit_data_basis.dropna(subset=['Au_ppm'] + covariates + phi_columns)

    deposit_data_list.append(deposit_data_basis)

## Comparison varying the levels of the basis function generating grid

In [8]:
dfs_names = ['3 levels', '3 levels no 0s', '2 levels', '2 levels no 0s', '1 level', '1 level no 0s']
for df, df_name in zip(deposit_data_list[1:], dfs_names[1:]):
    print(f"\nMetrics for df with {len(df.columns)} columns (grid with {df_name})")
    if df.empty:
        print(f"\nDataFrame for {df_name} is empty. Skipping...")
        continue
    trainer = DeepKrigingTrainer(df, regular_nn=False, plot_errors=False)
    trainer.train_neural_network()



Metrics for df with 3037 columns (grid with 3 levels no 0s)

Average Metrics Across Folds:
  Average MSE: 0.0008
  Average MAE: 0.0073
  Average Adjusted R2: 1.0042
  Average R2: 0.9000

Metrics for df with 7871 columns (grid with 2 levels)

Average Metrics Across Folds:
  Average MSE: 0.0006
  Average MAE: 0.0075
  Average Adjusted R2: 1.0015
  Average R2: 0.9015

Metrics for df with 1030 columns (grid with 2 levels no 0s)

Average Metrics Across Folds:
  Average MSE: 0.0007
  Average MAE: 0.0078
  Average Adjusted R2: 1.0145
  Average R2: 0.8922

Metrics for df with 1012 columns (grid with 1 level)

Average Metrics Across Folds:
  Average MSE: 0.0004
  Average MAE: 0.0060
  Average Adjusted R2: 1.0100
  Average R2: 0.9270

Metrics for df with 271 columns (grid with 1 level no 0s)

Average Metrics Across Folds:
  Average MSE: 0.0003
  Average MAE: 0.0058
  Average Adjusted R2: 1.0482
  Average R2: 0.9449


In [9]:
#Choose the last one
deposit_data_list[-1].to_csv('C:/Users/lin236/Documents/MEC_PC_Data/CSIRO Share/Export 19032024/Output/final_dataset_1_no_0.csv', index=False)
deposit_data_list[-2].to_csv('C:/Users/lin236/Documents/MEC_PC_Data/CSIRO Share/Export 19032024/Output/final_dataset_1_with_0.csv', index=False)